# K(α) Arc-Cosine Kernel Analysis

This notebook explores the theoretical K(α) kernel function from the thesis proposal.

The arc-cosine kernel describes the expected behavior of inner products under ReLU random projections:

$$K(\alpha) = \frac{1}{2\pi}(\sin\alpha + (\pi - \alpha)\cos\alpha)$$

**Topics:**
1. K(α) kernel visualization
2. Input-output angle relationship
3. Inner product transformation
4. Angle preservation analysis

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

from rp_study.analysis.kernel import (
    k_alpha,
    compute_output_angle,
    compute_kernel_values,
    plot_k_alpha,
    plot_angle_transformation,
    analyze_angle_preservation
)

## 1. The K(α) Kernel Function

The arc-cosine kernel arises from analyzing the expected inner product between two vectors after applying a random projection followed by ReLU activation.

In [ ]:
# Plot the K(alpha) function
fig = plot_k_alpha(n_points=400)
plt.show()

In [ ]:
# Key values of K(alpha)
alpha_vals = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]
alpha_names = ["0", "π/4", "π/2", "3π/4", "π"]

print("Key values of K(α):")
print("-" * 40)
for alpha, name in zip(alpha_vals, alpha_names):
    k_val = k_alpha(np.array([alpha]))[0]
    print(f"  K({name:5s}) = {k_val:.6f}")

print("\nInterpretation:")
print("  - K(0) = 0.5: identical vectors maintain high similarity")
print("  - K(π/2) ≈ 0.16: orthogonal vectors become weakly similar")
print("  - K(π) = 0: opposite vectors become orthogonal")

## 2. Input → Output Angle Transformation

How does the angle between two vectors change after ReLU projection?

In [ ]:
# Plot angle transformation
fig = plot_angle_transformation(n_points=400, d=100)
plt.show()

In [ ]:
# Detailed analysis
alpha = np.linspace(0.01, np.pi - 0.01, 100)  # Avoid boundary issues
output_angles = compute_output_angle(alpha)

# Plot the ratio theta_out / alpha
fig, ax = plt.subplots(figsize=(10, 5))

ratio = output_angles / alpha
ax.plot(alpha, ratio, 'b-', lw=2)
ax.axhline(y=1, color='gray', linestyle='--', label='No change (ratio=1)')

# Pi ticks
ticks = np.linspace(0, np.pi, 5)
labels = [r'$0$', r'$\frac{\pi}{4}$', r'$\frac{\pi}{2}$', r'$\frac{3\pi}{4}$', r'$\pi$']
ax.set_xticks(ticks)
ax.set_xticklabels(labels)

ax.set_xlabel(r'Input angle $\alpha$')
ax.set_ylabel(r'Ratio $\theta_{out}/\alpha$')
ax.set_title('Angle Contraction: Output Angle / Input Angle')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.2)

plt.tight_layout()
plt.show()

print(f"Average angle ratio: {np.mean(ratio):.3f}")
print(f"Minimum ratio: {np.min(ratio):.3f}")
print("Interpretation: ReLU projection contracts angles toward 0")

## 3. Inner Product Transformation

How does the inner product $\langle x, y \rangle$ transform under ReLU projection?

In [ ]:
# Compute kernel values for various d
alpha = np.linspace(0, np.pi, 400)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Raw inner product scaling with d
for d in [10, 100, 1000]:
    kernel_data = compute_kernel_values(alpha, d)
    axes[0].plot(alpha, kernel_data['raw_inner_product'], label=f'd={d}')

axes[0].set_xlabel(r'Input angle $\alpha$')
axes[0].set_ylabel(r'Expected inner product $d \cdot K(\alpha)$')
axes[0].set_title('Raw Inner Product (scales with d)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Normalized inner product (independent of d)
kernel_data = compute_kernel_values(alpha, 100)
axes[1].plot(kernel_data['input_cos'], kernel_data['k_alpha'], 'g-', lw=2)
axes[1].plot(kernel_data['input_cos'], kernel_data['input_cos'], 'k--', label='Identity')

axes[1].set_xlabel(r'Input similarity $\cos(\alpha)$')
axes[1].set_ylabel(r'Output similarity $K(\alpha)$')
axes[1].set_title('Input vs Output Similarity')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Angle Preservation Analysis

In [ ]:
# Analyze how well angles are preserved
alpha = np.linspace(0.1, np.pi - 0.1, 100)
analysis = analyze_angle_preservation(alpha, verbose=True)

In [ ]:
# Visualize the analysis
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Input vs Output angle
axes[0].plot(analysis['input_angles'], analysis['output_angles'], 'b-', lw=2)
axes[0].plot([0, np.pi], [0, np.pi], 'k--', label='Identity')
axes[0].set_xlabel('Input angle')
axes[0].set_ylabel('Output angle')
axes[0].set_title('Input vs Output Angle')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Angle difference
axes[1].plot(analysis['input_angles'], analysis['angle_difference'], 'r-', lw=2)
axes[1].axhline(y=0, color='gray', linestyle='--')
axes[1].set_xlabel('Input angle')
axes[1].set_ylabel('Angle difference (out - in)')
axes[1].set_title('Angle Change')
axes[1].grid(True, alpha=0.3)

# Contraction ratio
axes[2].plot(analysis['input_angles'], analysis['angle_ratio'], 'g-', lw=2)
axes[2].axhline(y=1, color='gray', linestyle='--', label='No contraction')
axes[2].set_xlabel('Input angle')
axes[2].set_ylabel('Contraction ratio')
axes[2].set_title('Angle Contraction Factor')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Multi-Layer Kernel Composition

What happens when we apply multiple layers of ReLU projections?

In [ ]:
def multi_layer_k_alpha(alpha, num_layers):
    """Compute the effective kernel after multiple layers."""
    # Start with input angle
    current_angle = alpha.copy()
    
    for _ in range(num_layers):
        # Compute output angle from current angle
        k_vals = k_alpha(current_angle)
        # Output angle = arccos(2*K(alpha))
        cos_out = np.clip(2 * k_vals, -1, 1)
        current_angle = np.arccos(cos_out)
    
    return current_angle

# Compute for different layer counts
alpha = np.linspace(0.01, np.pi - 0.01, 400)
layer_counts = [1, 2, 3, 5, 10, 20]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Output angles
for n_layers in layer_counts:
    output_angles = multi_layer_k_alpha(alpha, n_layers)
    axes[0].plot(alpha, output_angles, label=f'{n_layers} layers')

axes[0].set_xlabel('Input angle')
axes[0].set_ylabel('Output angle')
axes[0].set_title('Angle After Multiple ReLU Projection Layers')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Convergence: angle vs layer count for specific input angles
test_angles = [np.pi/6, np.pi/4, np.pi/3, np.pi/2, 2*np.pi/3]
angle_names = ['π/6', 'π/4', 'π/3', 'π/2', '2π/3']
max_layers = 50

for angle, name in zip(test_angles, angle_names):
    angles_over_layers = []
    current = angle
    for _ in range(max_layers):
        k_val = k_alpha(np.array([current]))[0]
        cos_out = np.clip(2 * k_val, -1, 1)
        current = np.arccos(cos_out)
        angles_over_layers.append(current)
    axes[1].plot(range(1, max_layers + 1), angles_over_layers, label=f'α₀={name}')

axes[1].set_xlabel('Number of layers')
axes[1].set_ylabel('Output angle')
axes[1].set_title('Angle Convergence with Depth')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

Key theoretical insights:

1. **K(α) kernel**: Describes expected inner product after ReLU projection
2. **Angle contraction**: Output angles are always smaller than input angles (ratio < 1)
3. **Similarity transformation**: High input similarity maps to high output similarity
4. **Multi-layer effect**: Repeated layers cause angles to converge toward a fixed point